# iSign Text-to-Pose — Data Preparation and Model Smoke Tests

This notebook reorganizes the code already written in the original working notebook into a single sequence: Google Drive/Hugging Face setup, dataset transfer and inspection, pilot-pair creation, compatibility patches, model construction, and tests.

**Scope:** this stops at the tests already performed. It does **not** add a training loop or claim that a useful signing model has been trained.

**Before running:** use Google Colab, authorize access to the gated iSign dataset on Hugging Face, and make sure your Google Drive has enough storage. The large-file transfer cell skips files that already exist and pass its checks. The first pilot uses only five examples, so it is for pipeline validation, not model-quality evaluation.


## 1. Mount Google Drive and create the project directory

In [ ]:
# CELL 1 — Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 2 — Create the iSign project directory on Drive.
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/ISL_Project/iSign")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset folder:", DATA_DIR)


## 2. Hugging Face setup and metadata files

In [ ]:
# CELL 3 — Install/upgrade the Hugging Face Hub client.
!pip install -U huggingface_hub


In [ ]:
# CELL 4 — Log in to Hugging Face so gated iSign files can be accessed.
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# CELL 5 — List files published in the iSign dataset repository.

from huggingface_hub import HfApi

api = HfApi()
repo_id = "Exploration-Lab/iSign"

info = api.dataset_info(
    repo_id,
    files_metadata=True
)

for f in info.siblings:
    size = getattr(f, "size", None)
    size_gb = f"{size / 1e9:.2f} GB" if size is not None else "size unknown"
    print(f"{size_gb:>12} | {f.rfilename}")


In [ ]:
# CELL 6 — Download the iSign CSV metadata files to Google Drive.
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil

DRIVE_DIR = Path("/content/drive/MyDrive/ISL_Project/iSign")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

repo_id = "Exploration-Lab/iSign"

files = [
    "iSign_v1.1.csv",
    "word-description-dataset_v1.1.csv",
    "word-presence-dataset_v1.1.csv",
]

for filename in files:
    downloaded = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset"
    )

    destination = DRIVE_DIR / filename
    shutil.copy2(downloaded, destination)
    print(f"Saved: {destination}")


In [ ]:
# CELL 7 — Inspect the downloaded CSV schemas and sample rows.
import pandas as pd

for filename in files:
    df = pd.read_csv(DRIVE_DIR / filename)
    print(f"\n--- {filename} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


In [ ]:
# CELL 8 — Read the dataset README from Hugging Face.
from huggingface_hub import hf_hub_download

readme_path = hf_hub_download(
    repo_id="Exploration-Lab/iSign",
    filename="README.md",
    repo_type="dataset"
)

with open(readme_path, "r", encoding="utf-8") as f:
    print(f.read())


In [ ]:
# CELL 9 — Inspect sequence IDs, video IDs, and text segments.
import pandas as pd

csv_path = "/content/drive/MyDrive/ISL_Project/iSign/iSign_v1.1.csv"
df = pd.read_csv(csv_path)

df["video_id"] = df["uid"].str.rsplit("-", n=1).str[0]
df["sequence_number"] = df["uid"].str.rsplit("-", n=1).str[-1]

print("Text segments:", len(df))
print("Unique videos:", df["video_id"].nunique())
print("\nExample segments:")
display(df[["uid", "video_id", "sequence_number", "text"]].head(10))


## 3. Transfer the large dataset parts to Drive

The following transfer cell was already present in the original notebook. It uploads the pose/video parts one at a time and checks for existing files. If the large files are already present and verified in Drive, you can skip this cell and proceed to the verification cell.


In [ ]:
# CELL 10 — Transfer the large gated iSign pose/video parts from Hugging Face to Drive.
# One-cell pipeline: download one file -> upload to Drive -> verify -> clear local cache -> next file

from pathlib import Path
import shutil
import time
from huggingface_hub import HfApi, hf_hub_download
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

REPO_ID = "Exploration-Lab/iSign"
REPO_TYPE = "dataset"
ROOT_NAME = "ISL_Project"
PROJECT_NAME = "iSign"
POSE_FOLDER_NAME = "pose_parts"
VIDEO_FOLDER_NAME = "video_parts"

POSE_FILES = [
    "iSign-poses_v1.1_part_aa",
    "iSign-poses_v1.1_part_ab",
    "iSign-poses_v1.1_part_ac",
    "iSign-poses_v1.1_part_ad",
]
VIDEO_FILES = [
    "iSign-videos_v1.1_part_aa",
    "iSign-videos_v1.1_part_ab",
]
FILES_TO_TRANSFER = [(name, POSE_FOLDER_NAME) for name in POSE_FILES] + [
    (name, VIDEO_FOLDER_NAME) for name in VIDEO_FILES
]

# Authenticate and initialize services. If already authenticated, Colab reuses the session.
auth.authenticate_user()
drive_service = build("drive", "v3")
hf_api = HfApi()

def find_folder(name, parent_id):
    q = (
        f"name = '{name}' and "
        "mimeType = 'application/vnd.google-apps.folder' and "
        "trashed = false"
    )
    if parent_id:
        q += f" and '{parent_id}' in parents"
    result = drive_service.files().list(
        q=q, fields="files(id,name,parents)", pageSize=100
    ).execute()
    matches = result.get("files", [])
    if matches:
        return matches[0]["id"]
    return None

def create_folder(name, parent_id):
    body = {
        "name": name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id],
    }
    result = drive_service.files().create(body=body, fields="id,name").execute()
    print(f"Created Drive folder: {name}")
    return result["id"]

def get_or_create_folder(name, parent_id):
    folder_id = find_folder(name, parent_id)
    return folder_id if folder_id else create_folder(name, parent_id)

def find_drive_file(name, folder_id):
    safe_name = name.replace("'", r"\'")
    q = (
        f"name = '{safe_name}' and '{folder_id}' in parents "
        "and trashed = false"
    )
    result = drive_service.files().list(
        q=q, fields="files(id,name,size,mimeType)", pageSize=100
    ).execute()
    return result.get("files", [])

def remove_cached_file(filename, downloaded_path=None):
    """Remove only this file's local HF snapshot link/blob after Drive verification."""
    cache_root = Path.home() / ".cache/huggingface/hub/datasets--Exploration-Lab--iSign"
    candidates = set()

    if downloaded_path:
        p = Path(downloaded_path)
        if p.exists() or p.is_symlink():
            candidates.add(p)
            try:
                resolved = p.resolve()
                if resolved != p:
                    candidates.add(resolved)
            except Exception:
                pass

    if cache_root.exists():
        for p in cache_root.rglob(filename):
            if p.is_file() or p.is_symlink():
                candidates.add(p)
                try:
                    resolved = p.resolve()
                    if resolved != p and cache_root in resolved.parents:
                        candidates.add(resolved)
                except Exception:
                    pass

    removed_bytes = 0
    for p in sorted(candidates, key=lambda x: len(str(x)), reverse=True):
        try:
            if p.is_symlink():
                p.unlink(missing_ok=True)
            elif p.is_file():
                size = p.stat().st_size
                p.unlink(missing_ok=True)
                removed_bytes += size
        except FileNotFoundError:
            pass
        except Exception as exc:
            print(f"Cache cleanup warning for {p}: {exc}")
    if removed_bytes:
        print(f"Freed about {removed_bytes / 1e9:.2f} GB from local cache.")
    return removed_bytes

# Pull authoritative file sizes from the dataset repository metadata.
print("Reading dataset file metadata from Hugging Face...")
repo_info = hf_api.dataset_info(REPO_ID, files_metadata=True)
repo_sizes = {
    item.rfilename: getattr(item, "size", None)
    for item in repo_info.siblings
}
missing = [name for name, _ in FILES_TO_TRANSFER if name not in repo_sizes]
if missing:
    raise RuntimeError(
        "Could not find these expected files in Hugging Face metadata: "
        + ", ".join(missing)
    )

# Resolve folders from My Drive instead of hardcoding folder IDs.
root_id = find_folder(ROOT_NAME, "root")
if not root_id:
    root_id = create_folder(ROOT_NAME, "root")
project_id = get_or_create_folder(PROJECT_NAME, root_id)
pose_folder_id = get_or_create_folder(POSE_FOLDER_NAME, project_id)
video_folder_id = get_or_create_folder(VIDEO_FOLDER_NAME, project_id)
folder_ids = {
    POSE_FOLDER_NAME: pose_folder_id,
    VIDEO_FOLDER_NAME: video_folder_id,
}

print("\nTransfer plan:")
total_bytes = sum(int(repo_sizes[name] or 0) for name, _ in FILES_TO_TRANSFER)
print(f"{len(FILES_TO_TRANSFER)} large files | metadata total ~{total_bytes / 1e9:.2f} GB")
print("Destination: My Drive / ISL_Project / iSign / [pose_parts or video_parts]")

for index, (filename, folder_name) in enumerate(FILES_TO_TRANSFER, start=1):
    expected_size = repo_sizes.get(filename)
    if expected_size is None:
        raise RuntimeError(f"File size unavailable for {filename}; stopping safely.")
    expected_size = int(expected_size)
    folder_id = folder_ids[folder_name]

    print(f"\n{'=' * 72}")
    print(f"[{index}/{len(FILES_TO_TRANSFER)}] {filename}")
    print(f"Expected size: {expected_size / 1e9:.2f} GB")

    # Check Drive first. If the exact-size file is already there, don't upload it again.
    existing_files = find_drive_file(filename, folder_id)
    matching = next(
        (f for f in existing_files if int(f.get("size", -1)) == expected_size),
        None
    )
    if matching:
        print(f"Already complete in Drive; skipping upload. Drive ID: {matching['id']}")
        # It is safe to clear the local cache copy only after confirming Drive size.
        remove_cached_file(filename)
        free_gb = shutil.disk_usage("/").free / 1e9
        print(f"Local free disk: {free_gb:.2f} GB")
        continue

    # Remove stale/incomplete same-name Drive entries before uploading a clean copy.
    for old in existing_files:
        print(
            f"Removing mismatched Drive entry ({old.get('size', 'unknown')} bytes) "
            f"before retry: {old['id']}"
        )
        drive_service.files().delete(fileId=old["id"]).execute()

    # If not cached, make sure there's enough disk for this single file plus safety margin.
    cache_root = Path.home() / ".cache/huggingface/hub/datasets--Exploration-Lab--iSign"
    cached_candidates = [
        p for p in cache_root.rglob(filename)
        if p.is_file() and p.stat().st_size == expected_size
    ] if cache_root.exists() else []

    downloaded_path = str(cached_candidates[0]) if cached_candidates else None
    if downloaded_path is None:
        free_bytes = shutil.disk_usage("/").free
        safety_margin = 6 * 1024**3
        required = expected_size + safety_margin
        if free_bytes < required:
            raise RuntimeError(
                f"Not enough Colab disk space for {filename}. "
                f"Free: {free_bytes / 1e9:.2f} GB; "
                f"needed about {required / 1e9:.2f} GB. "
                "No new download was started. Ensure previous verified cache files "
                "were cleared, then rerun this cell."
            )
        print("Downloading this file to the local HF cache (one file at a time)...")
        downloaded_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=filename,
            repo_type=REPO_TYPE,
        )
    else:
        print("Using existing complete cached file; no new download needed.")

    source = Path(downloaded_path)
    actual_source_size = source.stat().st_size
    if actual_source_size != expected_size:
        raise RuntimeError(
            f"Local file size mismatch for {filename}: "
            f"{actual_source_size} != expected {expected_size}. "
            "Keeping the local file for diagnosis."
        )

    print(f"Local source ready: {source}")
    print(f"Uploading to Drive folder '{folder_name}'...")
    media = MediaFileUpload(
        str(source),
        mimetype="application/octet-stream",
        chunksize=8 * 1024 * 1024,
        resumable=True,
    )
    request = drive_service.files().create(
        body={"name": filename, "parents": [folder_id]},
        media_body=media,
        fields="id,name,size",
    )

    response = None
    last_reported = -5
    start_time = time.time()
    while response is None:
        status, response = request.next_chunk(num_retries=5)
        if status:
            percent = int(status.progress() * 100)
            if percent >= last_reported + 5 or percent == 100:
                elapsed = max(time.time() - start_time, 1)
                sent_gb = expected_size * status.progress() / 1e9
                speed_mbps = (expected_size * status.progress() / elapsed) / 1e6
                print(
                    f"Upload: {percent:3d}% | {sent_gb:.2f}/{expected_size / 1e9:.2f} GB "
                    f"| average {speed_mbps:.1f} MB/s"
                )
                last_reported = percent

    uploaded_id = response["id"]
    uploaded_size = int(response.get("size", -1))

    # Verify with a fresh Drive metadata lookup, not only the upload response.
    verified = drive_service.files().get(
        fileId=uploaded_id, fields="id,name,size,mimeType,parents"
    ).execute()
    drive_size = int(verified.get("size", -1))
    if uploaded_size != expected_size or drive_size != expected_size:
        raise RuntimeError(
            f"Drive verification failed for {filename}. "
            f"Expected {expected_size}; upload response {uploaded_size}; "
            f"Drive metadata {drive_size}. Local cache has been retained."
        )

    print(f"VERIFIED: {filename} | {drive_size} bytes | Drive ID: {uploaded_id}")

    # Only delete local cached bytes after exact-size Drive verification.
    remove_cached_file(filename, downloaded_path)
    print(f"Finished {filename}. Local free disk: {shutil.disk_usage('/').free / 1e9:.2f} GB")

print("\nALL REQUESTED LARGE FILES ARE VERIFIED IN GOOGLE DRIVE.")
print("Pose destination:", f"ISL_Project/iSign/{POSE_FOLDER_NAME}")
print("Video destination:", f"ISL_Project/iSign/{VIDEO_FOLDER_NAME}")


In [ ]:
# CELL 11 — Verify which iSign files are present in Google Drive.

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path("/content/drive/MyDrive/ISL_Project/iSign")

for folder in [BASE, BASE / "pose_parts", BASE / "video_parts"]:
    print(f"\n{folder}")
    if folder.exists():
        for f in sorted(folder.iterdir()):
            if f.is_file():
                print(f"  {f.name}: {f.stat().st_size / 1e9:.2f} GB")
    else:
        print("  Folder not found")


## 4. Clone the Progressive Transformers repository and install the pose reader

In [ ]:
# CELL 12 — Clone the existing ProgressiveTransformersSLP repository and inspect its starting data/config files.

%cd /content
!git clone --depth 1 https://github.com/BenSaunders27/ProgressiveTransformersSLP.git
%cd /content/ProgressiveTransformersSLP

print("\n=== Model configuration ===")
!cat Configs/Base.yaml

print("\n=== Example training data files ===")
!find Data/tmp -maxdepth 2 -type f


In [ ]:
# CELL 13 — Install pose-format for reading iSign .pose files.

%pip install pose-format


In [ ]:
# CELL 14 — Check the pose archive parts and confirm they are not independently readable ZIP files.

from pathlib import Path
import zipfile

root = Path("/content/drive/MyDrive/ISL_Project/iSign/pose_parts")
parts = [
    root / "iSign-poses_v1.1_part_aa",
    root / "iSign-poses_v1.1_part_ab",
    root / "iSign-poses_v1.1_part_ac",
    root / "iSign-poses_v1.1_part_ad",
]

for p in parts:
    with open(p, "rb") as f:
        header = f.read(16)
    print(p.name, "| bytes:", p.stat().st_size, "| header:", header.hex(" "))

print("\nZIP detection for first part:")
try:
    with zipfile.ZipFile(parts[0]) as z:
        print("Readable as standalone ZIP")
        print("First entries:", z.namelist()[:5])
except Exception as e:
    print(type(e).__name__ + ":", str(e)[:250])


## 5. Read the split archive without creating a huge combined copy

In [ ]:
# CELL 15 — Define VirtualConcatFile and the ordered pose-part paths. This exposes the split archive as one seekable file without writing a combined archive.

import io
import zipfile
from pathlib import Path

root = Path("/content/drive/MyDrive/ISL_Project/iSign/pose_parts")

paths = [
    root / "iSign-poses_v1.1_part_aa",
    root / "iSign-poses_v1.1_part_ab",
    root / "iSign-poses_v1.1_part_ac",
    root / "iSign-poses_v1.1_part_ad",
]

class VirtualConcatFile(io.BufferedIOBase):
    """Seekable view of multiple files in concatenated order."""
    def __init__(self, paths):
        self.paths = paths
        self.files = [open(p, "rb") for p in paths]
        self.sizes = [p.stat().st_size for p in paths]
        self.starts = []
        total = 0
        for size in self.sizes:
            self.starts.append(total)
            total += size
        self.total_size = total
        self.pos = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            new_pos = offset
        elif whence == io.SEEK_CUR:
            new_pos = self.pos + offset
        elif whence == io.SEEK_END:
            new_pos = self.total_size + offset
        else:
            raise ValueError("Invalid whence")
        if new_pos < 0:
            raise ValueError("Negative seek position")
        self.pos = new_pos
        return self.pos

    def read(self, size=-1):
        if self.pos >= self.total_size:
            return b""
        if size is None or size < 0:
            size = self.total_size - self.pos
        size = min(size, self.total_size - self.pos)
        chunks = []

        while size > 0:
            part_idx = max(
                i for i, start in enumerate(self.starts)
                if start <= self.pos
            )
            local_pos = self.pos - self.starts[part_idx]
            available = self.sizes[part_idx] - local_pos
            take = min(size, available)

            f = self.files[part_idx]
            f.seek(local_pos)
            chunk = f.read(take)
            if not chunk:
                break

            chunks.append(chunk)
            self.pos += len(chunk)
            size -= len(chunk)

        return b"".join(chunks)

    def close(self):
        for f in self.files:
            f.close()
        super().close()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

print("Opening virtual archive view; no combined archive will be written.")
with VirtualConcatFile(paths) as virtual_file:
    with zipfile.ZipFile(virtual_file, "r") as archive:
        print("Archive opened successfully.")
        print("Number of entries:", len(archive.namelist()))
        print("First 15 pose entries:")
        pose_names = [
            name for name in archive.namelist()
            if name.lower().endswith(".pose")
        ]
        print("\n".join(pose_names[:15]))


In [ ]:
# CELL 16 — Load the text CSV, list pose entries, and match CSV UIDs to .pose filenames.

import pandas as pd
import zipfile
from pathlib import PurePosixPath

csv_path = "/content/drive/MyDrive/ISL_Project/iSign/iSign_v1.1.csv"
df = pd.read_csv(csv_path)

with VirtualConcatFile(paths) as virtual_file:
    with zipfile.ZipFile(virtual_file, "r") as archive:
        pose_names = [
            name for name in archive.namelist()
            if name.lower().endswith(".pose")
        ]

        # Compare CSV identifiers with archive filenames.
        pose_by_stem = {
            PurePosixPath(name).stem: name
            for name in pose_names
        }

        print("CSV sample:")
        print(df.head(10).to_string(index=False))

        print("\nArchive filename sample:")
        print("\n".join(pose_names[:10]))

        print("\nExact UID-to-filename matches:")
        matches = 0
        for uid in df["uid"].head(20).astype(str):
            match = pose_by_stem.get(uid)
            print(uid, "->", match)
            matches += match is not None

        print(f"\nExact matches among first 20: {matches}/20")


In [ ]:
# CELL 17 — Read one pose example and inspect its tensor shapes and basic values.

from pose_format import Pose
import numpy as np
import zipfile

with VirtualConcatFile(paths) as virtual_file:
    with zipfile.ZipFile(virtual_file, "r") as archive:
        pose_name = next(
            name for name in archive.namelist()
            if name.lower().endswith(".pose")
        )

        print("Sample:", pose_name)
        pose_bytes = archive.read(pose_name)
        print("File size:", len(pose_bytes), "bytes")

        pose = Pose.read(pose_bytes)

        body = pose.body
        data = np.asarray(body.data)
        confidence = np.asarray(body.confidence)

        print("\n=== Pose structure ===")
        print("Data shape:", data.shape)
        print("Confidence shape:", confidence.shape)
        print("FPS:", getattr(pose.header, "fps", "not exposed"))
        print("Body type:", type(body).__name__)
        print("Data dtype:", data.dtype)
        print("First values:", data.reshape(-1)[:20])


In [ ]:
# CELL 18 — Inspect coordinate groups, coordinate ranges, and confidence/zero-filled rates.

import numpy as np

print("=== Header dimensions ===")
print(vars(pose.header.dimensions))

coords = np.asarray(pose.body.data)[:, 0, :, :]
conf = np.asarray(pose.body.confidence)[:, 0, :]

groups = {
    "body_screen": (0, 33),
    "face": (33, 501),
    "left_hand": (501, 522),
    "right_hand": (522, 543),
    "body_world": (543, 576),
}

print("\n=== Zero-coordinate and low-confidence rates ===")
for name, (start, end) in groups.items():
    xyz = coords[:, start:end, :]
    scores = conf[:, start:end]

    # Count a landmark as zero-filled only if all 3 coordinates are zero.
    zero_rate = np.all(np.isclose(xyz, 0), axis=-1).mean()
    low_conf_rate = (scores < 0.5).mean()

    print(
        f"{name:12s} | "
        f"all-zero landmarks: {zero_rate:.1%} | "
        f"confidence < 0.5: {low_conf_rate:.1%}"
    )

print("\n=== First-frame hand landmarks ===")
for name, idx in [("left hand", 501), ("right hand", 522)]:
    print(name, "first landmark:", coords[0, idx].tolist(),
          "| confidence:", float(conf[0, idx]))


In [ ]:
# CELL 19 — Open the virtual archive for the pilot conversion cell below.

import zipfile
from pathlib import Path

# Reuse the existing virtual concatenation class and part paths.
virtual_file = VirtualConcatFile(paths)
archive = zipfile.ZipFile(virtual_file, "r")

print("Archive reopened:", len(archive.namelist()), "entries")
print("Archive is closed:", archive.fp is None)


In [ ]:
# CELL 20 — Create the five-sequence text/skeleton pilot using 75 body/hand landmarks, normalized X/Y, confidence filtering, and a frame counter.

import numpy as np
from pathlib import Path
from pose_format import Pose

out_dir = Path("/content/isl_phase2_pilot")
out_dir.mkdir(parents=True, exist_ok=True)

selected = df.head(5)
text_lines = []
skeleton_lines = []
sequence_ids = []

for _, row in selected.iterrows():
    uid = str(row["uid"])
    text = str(row["text"]).strip().lower()

    # Find the matching pose archive entry.
    pose_name = pose_by_stem.get(uid)
    if pose_name is None:
        raise ValueError(f"No pose file found for {uid}")

    pose = Pose.read(archive.read(pose_name))
    coords = np.asarray(pose.body.data, dtype=np.float32)[:, 0, :, :]
    conf = np.asarray(pose.body.confidence, dtype=np.float32)[:, 0, :]

    # 33 screen-space body landmarks + 21 left-hand + 21 right-hand.
    indices = np.r_[0:33, 501:543]
    xyz = coords[:, indices, :].copy()
    scores = conf[:, indices]

    # Normalize image X/Y from approximately [0, 300] to [-1, 1].
    xyz[:, :, 0] = xyz[:, :, 0] / 150.0 - 1.0
    xyz[:, :, 1] = xyz[:, :, 1] / 150.0 - 1.0

    # Mask missing or low-confidence landmarks so that zero-filled
    # missing points do not become artificial corner coordinates.
    xyz[scores < 0.5] = 0.0

    # Flatten each frame to 225 coordinates, then append a frame counter.
    n_frames = len(xyz)
    counter = np.arange(n_frames, dtype=np.float32) / n_frames
    frames = np.concatenate(
        [xyz.reshape(n_frames, 225), counter[:, None]], axis=1
    )

    # The repository expects one flattened sequence per line.
    skeleton_lines.append(
        " ".join(f"{v:.5f}" for v in frames.reshape(-1))
    )
    text_lines.append(text)
    sequence_ids.append(uid)

(out_dir / "train.text").write_text("\n".join(text_lines) + "\n")
(out_dir / "train.skels").write_text("\n".join(skeleton_lines) + "\n")
(out_dir / "train.files").write_text("\n".join(sequence_ids) + "\n")

print("Pilot created:", out_dir)
print("Sequences:", len(sequence_ids))
for uid, line in zip(sequence_ids, skeleton_lines):
    values = line.split()
    print(
        uid,
        "| frames:", len(values) // 226,
        "| values:", len(values),
        "| valid frame width:", len(values) % 226 == 0
    )


## 6. Patch legacy imports and create a native PyTorch data loader

The original repository depends on old `torchtext` APIs that are unavailable in the current runtime. These cells apply the compatibility edits that were performed during the session. They modify files in the cloned repository; they do not start training.


In [ ]:
# CELL 21 — Remove the obsolete torchtext Dataset type annotation from vocabulary.py.

%cd /content/ProgressiveTransformersSLP

from pathlib import Path

p = Path("vocabulary.py")
source = p.read_text()

# Remove the legacy Dataset type annotation from build_vocab.
source = source.replace(
    "def build_vocab(field: str, max_size: int, min_freq: int, dataset: Dataset,",
    "def build_vocab(field: str, max_size: int, min_freq: int, dataset,"
)

p.write_text(source)
print("Updated vocabulary.py")


In [ ]:
# CELL 22 — Remove the obsolete torchtext Dataset import from helpers.py.

%cd /content/ProgressiveTransformersSLP

from pathlib import Path
import shutil

p = Path("helpers.py")
backup = Path("helpers.py.pre_compat.bak")

if not backup.exists():
    shutil.copy2(p, backup)

source = p.read_text()
source = source.replace(
    "from torchtext.data import Dataset\n",
    ""
)
p.write_text(source)

print("Removed legacy Dataset import from helpers.py")


In [ ]:
# CELL 23 — Create isign_data.py with a PyTorch Dataset, vocabulary builder, and batch loader.

%cd /content/ProgressiveTransformersSLP

from pathlib import Path

Path("isign_data.py").write_text(r'''
from pathlib import Path
from types import SimpleNamespace
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
from constants import UNK_TOKEN, PAD_TOKEN, BOS_TOKEN, EOS_TOKEN
from vocabulary import Vocabulary

class ISignDataset(Dataset):
    def __init__(self, prefix, trg_size=225):
        prefix = Path(prefix)
        texts = (Path(str(prefix) + ".text")).read_text().splitlines()
        skels = (Path(str(prefix) + ".skels")).read_text().splitlines()
        files = (Path(str(prefix) + ".files")).read_text().splitlines()

        if not (len(texts) == len(skels) == len(files)):
            raise ValueError("Text, skeleton, and file counts differ.")

        self.examples = []
        self.trg_size = trg_size + 1

        for text, skeleton, uid in zip(texts, skels, files):
            values = [float(v) for v in skeleton.split()]
            if not values or len(values) % self.trg_size:
                raise ValueError(f"Invalid frame width for {uid}")
            frames = torch.tensor(values, dtype=torch.float32)
            frames = frames.reshape(-1, self.trg_size)
            self.examples.append({
                "src": text.lower().split(),
                "trg": frames,
                "file_paths": uid,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def build_isign_vocab(dataset):
    counter = Counter()
    for ex in dataset.examples:
        counter.update(ex["src"])
    tokens = sorted(counter, key=lambda t: (-counter[t], t))
    return Vocabulary(tokens=tokens)

def collate_isign(examples, vocab):
    pad_id = vocab.stoi[PAD_TOKEN]
    src_seqs = [
        torch.tensor([vocab.stoi[t] for t in ex["src"]] + [vocab.stoi[EOS_TOKEN]],
                     dtype=torch.long)
        for ex in examples
    ]
    src_lengths = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)
    src = torch.nn.utils.rnn.pad_sequence(
        src_seqs, batch_first=True, padding_value=pad_id
    )

    max_frames = max(ex["trg"].shape[0] for ex in examples)
    trg = torch.zeros(len(examples), max_frames, examples[0]["trg"].shape[1])
    for i, ex in enumerate(examples):
        trg[i, :ex["trg"].shape[0]] = ex["trg"]

    return SimpleNamespace(
        src=(src, src_lengths),
        trg=trg,
        file_paths=[ex["file_paths"] for ex in examples],
    )

def make_isign_loader(dataset, vocab, batch_size=2, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=lambda examples: collate_isign(examples, vocab),
    )
''')

print("Created isign_data.py")


In [ ]:
# CELL 24 — Update the collator to include true frame_lengths for padding masks.

%cd /content/ProgressiveTransformersSLP

from pathlib import Path
import shutil

p = Path("isign_data.py")
backup = Path("isign_data.py.pre_mask.bak")

if not backup.exists():
    shutil.copy2(p, backup)

source = p.read_text()
start = source.index("def collate_isign(")
end = source.index("\ndef make_isign_loader(", start)

new_collator = '''
def collate_isign(examples, vocab):
    pad_id = vocab.stoi[PAD_TOKEN]

    src_seqs = [
        torch.tensor(
            [vocab.stoi[t] for t in ex["src"]] + [vocab.stoi[EOS_TOKEN]],
            dtype=torch.long
        )
        for ex in examples
    ]
    src_lengths = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)
    src = torch.nn.utils.rnn.pad_sequence(
        src_seqs, batch_first=True, padding_value=pad_id
    )

    frame_lengths = torch.tensor(
        [ex["trg"].shape[0] for ex in examples], dtype=torch.long
    )
    max_frames = int(frame_lengths.max().item())
    frame_width = examples[0]["trg"].shape[1]

    trg = torch.zeros(len(examples), max_frames, frame_width)
    for i, ex in enumerate(examples):
        trg[i, :ex["trg"].shape[0]] = ex["trg"]

    return SimpleNamespace(
        src=(src, src_lengths),
        trg=trg,
        frame_lengths=frame_lengths,
        file_paths=[ex["file_paths"] for ex in examples],
    )
'''

p.write_text(source[:start] + new_collator + source[end:])
print("Updated isign_data.py successfully.")


In [ ]:
# CELL 25 — Test the native loader and confirm the target width is 225 coordinates plus one counter.

from isign_data import ISignDataset, build_isign_vocab, make_isign_loader

pilot = ISignDataset("/content/isl_phase2_pilot/train", trg_size=225)
vocab = build_isign_vocab(pilot)
loader = make_isign_loader(pilot, vocab, batch_size=2)

batch = next(iter(loader))

print("Dataset size:", len(pilot))
print("Vocabulary size:", len(vocab))
print("Source shape:", batch.src[0].shape)
print("Source lengths:", batch.src[1].tolist())
print("Target shape:", batch.trg.shape)
print("Target frame width:", batch.trg.shape[-1])
print("File paths:", batch.file_paths)

assert batch.trg.shape[-1] == 226
assert batch.src[0].shape[0] == 2
print("Basic loader test passed.")


## 7. Create the pilot configuration and patch loss/batch masking

In [ ]:
# CELL 26 — Create a separate pilot YAML configuration with text input, trg_size=225, batch size 2, and CPU-mode setting for the initial tests.

%cd /content/ProgressiveTransformersSLP

from pathlib import Path
import yaml

config = yaml.safe_load(Path("Configs/Base.yaml").read_text())

config["data"]["src"] = "text"
config["data"]["train"] = "/content/isl_phase2_pilot/train"
config["data"]["dev"] = "/content/isl_phase2_pilot/train"
config["data"]["test"] = "/content/isl_phase2_pilot/train"
config["data"].pop("src_vocab", None)

config["model"]["trg_size"] = 225
config["training"]["use_cuda"] = False
config["training"]["batch_size"] = 2
config["training"]["epochs"] = 1
config["training"]["continue"] = False
config["training"]["model_dir"] = "/content/isl_phase2_pilot/model"

pilot_config_path = Path("/content/isl_phase2_pilot/config.yaml")
pilot_config_path.write_text(yaml.safe_dump(config, sort_keys=False))

print(pilot_config_path)
print("Source:", config["data"]["src"])
print("Target coordinates:", config["model"]["trg_size"])
print("Expected frame width:", config["model"]["trg_size"] + 1)
print("CUDA enabled:", config["training"]["use_cuda"])


In [ ]:
# CELL 27 — Replace RegLoss with an explicit frame-mask option so valid zero coordinates are not mistaken for padding.

%cd /content/ProgressiveTransformersSLP

from pathlib import Path
import shutil

p = Path("loss.py")
backup = Path("loss.py.pre_mask.bak")

if not backup.exists():
    shutil.copy2(p, backup)

source = p.read_text()
start = source.index("class RegLoss(")
end = source.index("\nclass ", start + 1)

new_class = '''
class RegLoss(nn.Module):
    """Regression loss with optional explicit valid-frame masking."""

    def __init__(self, cfg, target_pad=0.0):
        super().__init__()
        self.loss = cfg["training"]["loss"].lower()
        self.target_pad = target_pad
        self.loss_scale = cfg["model"].get("loss_scale", 1.0)

        if self.loss == "mse":
            self.criterion = nn.MSELoss(reduction="none")
        elif self.loss == "l1":
            self.criterion = nn.L1Loss(reduction="none")
        else:
            raise ValueError(f"Unsupported regression loss: {self.loss}")

    def forward(self, preds, targets, frame_mask=None):
        element_loss = self.criterion(preds, targets)

        if frame_mask is None:
            # Preserve legacy behavior when no explicit mask is supplied.
            mask = (targets != self.target_pad)
        else:
            # frame_mask must be [batch, time]; broadcast over coordinates.
            if frame_mask.ndim != 2:
                raise ValueError("frame_mask must have shape [batch, time]")
            if tuple(frame_mask.shape) != tuple(targets.shape[:2]):
                raise ValueError(
                    f"Mask shape {tuple(frame_mask.shape)} does not match "
                    f"target time dimensions {tuple(targets.shape[:2])}"
                )
            mask = frame_mask.to(
                device=targets.device, dtype=torch.bool
            ).unsqueeze(-1).expand_as(targets)

        mask = mask.to(dtype=element_loss.dtype)
        denominator = mask.sum().clamp_min(1.0)
        loss = (element_loss * mask).sum() / denominator
        return loss * self.loss_scale
'''

p.write_text(source[:start] + new_class + "\n")
print("Updated RegLoss with explicit frame-mask support.")


In [ ]:
# CELL 28 — Ensure loss.py imports torch and reload the updated loss module.

%cd /content/ProgressiveTransformersSLP

from pathlib import Path
import importlib
import loss

p = Path("loss.py")
source = p.read_text()

if "import torch\n" not in source:
    source = source.replace(
        "import torch.nn as nn",
        "import torch\nimport torch.nn as nn"
    )
    p.write_text(source)

importlib.reload(loss)
print("torch imported in loss.py")
print("Loss signature:", __import__("inspect").signature(loss.RegLoss.forward))


In [ ]:
# CELL 29 — Patch batch.py to build the decoder padding mask from frame lengths, not coordinate values.

%cd /content/ProgressiveTransformersSLP

from pathlib import Path
import shutil

p = Path("batch.py")
backup = Path("batch.py.pre_frame_mask.bak")

if not backup.exists():
    shutil.copy2(p, backup)

source = p.read_text()

old = '''            # Target Pad is dynamic, so we exclude the padded areas from the loss computation
            trg_mask = (self.trg_input != self.target_pad).unsqueeze(1)
            # This increases the shape of the target mask to be even (16,1,120,120) -
            # adding padding that replicates - so just continues the False's or True's
            pad_amount = self.trg_input.shape[1] - self.trg_input.shape[2]
            # Create the target mask the same size as target input
            self.trg_mask = (F.pad(input=trg_mask.double(), pad=(pad_amount, 0, 0, 0), mode='replicate') == 1.0)
'''

new = '''            # Build decoder padding mask from actual sequence lengths,
            # never from coordinate values (zero is a valid coordinate).
            if hasattr(torch_batch, "frame_lengths"):
                input_lengths = (torch_batch.frame_lengths - 1).clamp_min(0)
                steps = torch.arange(self.trg_input.shape[1])
                valid_steps = steps.unsqueeze(0) < input_lengths.unsqueeze(1)
                self.trg_mask = valid_steps.unsqueeze(1)
            else:
                # Legacy compatibility for datasets without frame lengths.
                valid_steps = torch.ones(
                    self.trg_input.shape[:2], dtype=torch.bool
                )
                self.trg_mask = valid_steps.unsqueeze(1)
'''

if old not in source:
    raise ValueError(
        "Expected original mask block was not found. "
        "No changes were written; inspect batch.py before proceeding."
    )

p.write_text(source.replace(old, new))
print("Updated batch.py to use frame-length masks.")


## 8. Build the model and run the recorded validation tests

In [ ]:
# CELL 30 — Build the Transformer using the pilot configuration and assert 226-value input/output frames.

%cd /content/ProgressiveTransformersSLP

import yaml
from pathlib import Path
from vocabulary import Vocabulary
from constants import UNK_TOKEN, PAD_TOKEN, BOS_TOKEN, EOS_TOKEN
from isign_data import ISignDataset, build_isign_vocab
from model import build_model

cfg = yaml.safe_load(
    Path("/content/isl_phase2_pilot/config.yaml").read_text()
)

pilot = ISignDataset("/content/isl_phase2_pilot/train", trg_size=225)
src_vocab = build_isign_vocab(pilot)

# Continuous pose targets use numeric vectors, not token IDs.
trg_vocab = Vocabulary(tokens=[])

model = build_model(cfg, src_vocab=src_vocab, trg_vocab=trg_vocab)
model.eval()

print("Model built successfully")
print("Source vocabulary:", len(src_vocab))
print("Target input width:", model.in_trg_size)
print("Target output width:", model.out_trg_size)
print("Decoder output width:", model.decoder.output_layer.out_features)

assert model.in_trg_size == 226
assert model.out_trg_size == 226
assert model.decoder.output_layer.out_features == 226

print("Model dimension test passed.")


In [ ]:
# CELL 31 — Run the teacher-forced forward pass and check prediction/target shapes and finite values.

%cd /content/ProgressiveTransformersSLP

import torch
from batch import Batch
from isign_data import make_isign_loader
from loss import RegLoss

loader = make_isign_loader(pilot, src_vocab, batch_size=2)
raw_batch = next(iter(loader))

# Create the batch object expected by the existing model.
batch = Batch(
    torch_batch=raw_batch,
    pad_index=src_vocab.stoi["<pad>"],
    model=model,
)

# Run the model's teacher-forced forward pass.
model.eval()
with torch.no_grad():
    prediction, _ = model(
        src=batch.src,
        trg_input=batch.trg_input,
        src_mask=batch.src_mask,
        src_lengths=batch.src_lengths,
        trg_mask=batch.trg_mask,
    )

print("Prediction shape:", prediction.shape)
print("Target shape:", batch.trg.shape)
print("Prediction finite:", torch.isfinite(prediction).all().item())
print("Target finite:", torch.isfinite(batch.trg).all().item())

assert prediction.shape == batch.trg.shape
assert torch.isfinite(prediction).all()
print("Forward-pass test passed.")


In [ ]:
# CELL 32 — Test masked regression loss in isolation, including valid zero coordinates and zero gradient on a padded frame.

import torch
import loss

criterion = loss.RegLoss(cfg)

preds = torch.tensor(
    [[[1.0, 2.0], [3.0, 4.0], [100.0, 100.0]]],
    requires_grad=True
)
targets = torch.tensor(
    [[[0.0, 2.0], [3.0, 0.0], [0.0, 0.0]]]
)
frame_mask = torch.tensor([[True, True, False]])

value = criterion(preds, targets, frame_mask=frame_mask)
value.backward()

print("Masked loss:", value.item())
print("Gradient finite:", torch.isfinite(preds.grad).all().item())
print("Padded-frame gradient:", preds.grad[0, 2].tolist())

assert torch.isfinite(value)
assert torch.isfinite(preds.grad).all()
assert torch.all(preds.grad[0, 2] == 0)
print("Isolated masked-loss test passed.")


In [ ]:
# CELL 33 — Reload isign_data.py and verify the explicit frame-length mask.

import importlib
import isign_data

importlib.reload(isign_data)

loader = isign_data.make_isign_loader(
    pilot, src_vocab, batch_size=2
)
raw_batch = next(iter(loader))

frame_lengths = raw_batch.frame_lengths
max_frames = raw_batch.trg.shape[1]

valid_frames = (
    torch.arange(max_frames).unsqueeze(0)
    < frame_lengths.unsqueeze(1)
)

print("Target shape:", raw_batch.trg.shape)
print("Frame lengths:", frame_lengths.tolist())
print("Valid-frame mask shape:", valid_frames.shape)
print("Valid frames:", valid_frames.sum(dim=1).tolist())

assert valid_frames.sum(dim=1).tolist() == frame_lengths.tolist()
print("Frame-mask test passed.")


In [ ]:
# CELL 34 — Reload batch.py and verify the decoder mask matches the true target lengths.

import importlib
import batch as batch_module
import isign_data
import torch

importlib.reload(batch_module)
Batch = batch_module.Batch

loader = isign_data.make_isign_loader(pilot, src_vocab, batch_size=2)
raw_batch = next(iter(loader))

test_batch = Batch(
    torch_batch=raw_batch,
    pad_index=src_vocab.stoi["<pad>"],
    model=model,
)

expected_lengths = (raw_batch.frame_lengths - 1).clamp_min(0)
mask_steps = test_batch.trg_mask.squeeze(1)

print("Decoder mask shape:", tuple(test_batch.trg_mask.shape))
print("Expected valid steps:", expected_lengths.tolist())
print("Actual valid steps:", mask_steps.sum(dim=1).tolist())

assert mask_steps.sum(dim=1).tolist() == expected_lengths.tolist()
print("Decoder mask test passed.")


In [ ]:
# CELL 35 — Run the full forward/backward test with masked loss and check that gradients are finite.

import torch
import loss
import isign_data

loader = isign_data.make_isign_loader(pilot, src_vocab, batch_size=2)
raw_batch = next(iter(loader))

batch = Batch(
    torch_batch=raw_batch,
    pad_index=src_vocab.stoi["<pad>"],
    model=model,
)

target_lengths = (raw_batch.frame_lengths - 1).clamp_min(0)
valid_target_frames = (
    torch.arange(batch.trg.shape[1]).unsqueeze(0)
    < target_lengths.unsqueeze(1)
)

criterion = loss.RegLoss(cfg)
model.train()
model.zero_grad(set_to_none=True)

prediction, _ = model(
    src=batch.src,
    trg_input=batch.trg_input,
    src_mask=batch.src_mask,
    src_lengths=batch.src_lengths,
    trg_mask=batch.trg_mask,
)

loss_value = criterion(
    prediction, batch.trg, frame_mask=valid_target_frames
)
loss_value.backward()

gradients_finite = all(
    p.grad is None or torch.isfinite(p.grad).all().item()
    for p in model.parameters()
)

print("Prediction shape:", tuple(prediction.shape))
print("Target shape:", tuple(batch.trg.shape))
print("Masked loss:", loss_value.item())
print("Loss finite:", torch.isfinite(loss_value).item())
print("Gradients finite:", gradients_finite)

assert prediction.shape == batch.trg.shape
assert torch.isfinite(loss_value)
assert gradients_finite
print("Full backward test passed.")


## 9. Verify the current GPU runtime

This is the GPU verification cell already run in the original notebook. It reports the active PyTorch build, CUDA availability, and GPU name. No training loop is included in this cleaned notebook.


In [ ]:
# CELL 36 — Verify CUDA and identify the active GPU.
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
